# MotoGP Grand Prix Race Winners - Dataset Exploration

This notebook explores the race winners dataset covering all Grand Prix races across different circuits, classes, constructors, riders, and seasons.

## 0. Notebook Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading

Let's load the grand_prix_race_winners.csv file containing race results data.

In [ ]:
# Load data from CSV
data_path = Path("../data/raw/grand_prix_race_winners.csv")
df = pd.read_csv(data_path)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

## 2. Data Structure Exploration

Let's examine the data structure and understand the scope of the dataset.

In [ ]:
print(f"Dimensions: {df.shape}")
print("-" * 50)
print(f"Columns: {list(df.columns)}")
print("-" * 50)
print(f"\nData types:")
print(df.dtypes)
print("-" * 50)
print(f"\nNull values:")
print(df.isnull().sum())

In [ ]:
# Unique values in each column
print("=== UNIQUE VALUES ANALYSIS ===")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"\n{col}: {unique_count} unique values")
    if unique_count <= 15:  # Show values if there are few
        print(f"Values: {sorted(df[col].unique())}")
    else:
        print(f"Sample: {sorted(df[col].unique())[:10]}...")

In [ ]:
# Time period analysis
print("=== TEMPORAL COVERAGE ===")
print(f"Earliest season: {df['Season'].min()}")
print(f"Latest season: {df['Season'].max()}")
print(f"Total seasons: {df['Season'].nunique()}")
print(f"Total races: {len(df)}")
print(f"Average races per season: {len(df) / df['Season'].nunique():.1f}")

## 3. Classes and Categories Analysis

Let's analyze the distribution across different racing classes.

In [ ]:
# Classes distribution
print("=== RACING CLASSES ANALYSIS ===")
class_counts = df['Class'].value_counts()
print("Races per class:")
print(class_counts)

# Pie chart of classes
plt.figure(figsize=(10, 8))
plt.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', 
        startangle=90)
plt.title('Distribution of Races by Class')
plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Classes evolution over time
print("=== CLASS EVOLUTION OVER TIME ===")
class_timeline = df.groupby(['Season', 'Class']).size().unstack(fill_value=0)

plt.figure(figsize=(15, 8))
for class_name in class_timeline.columns:
    plt.plot(class_timeline.index, class_timeline[class_name], 
             marker='o', linewidth=2, label=class_name)

plt.xlabel('Season')
plt.ylabel('Number of Races')
plt.title('Evolution of Racing Classes Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show recent years for better visibility
recent_years = class_timeline.tail(10)
print("\nRaces per class in recent years (2013-2022):")
print(recent_years)

## 4. Constructor Analysis

Let's analyze constructor performance across classes and time.

In [ ]:
# Top constructors by total wins
print("=== TOP CONSTRUCTORS BY TOTAL WINS ===")
constructor_wins = df['Constructor'].value_counts()
print("Top 15 constructors:")
print(constructor_wins.head(15))

# Bar chart of top 15 constructors
plt.figure(figsize=(14, 8))
top_15_constructors = constructor_wins.head(15)
plt.bar(range(len(top_15_constructors)), top_15_constructors.values, color='lightblue')
plt.xticks(range(len(top_15_constructors)), top_15_constructors.index, rotation=45)
plt.xlabel('Constructor')
plt.ylabel('Number of Wins')
plt.title('Top 15 Constructors by Total Wins (All Classes)')
plt.tight_layout()
plt.show()

In [ ]:
# Constructor performance by class
print("=== CONSTRUCTOR PERFORMANCE BY CLASS ===")
constructor_class = df.groupby(['Class', 'Constructor']).size().unstack(fill_value=0)

# Top constructors per class
for class_name in df['Class'].unique():
    class_data = df[df['Class'] == class_name]
    top_constructors = class_data['Constructor'].value_counts().head(5)
    print(f"\nTop 5 constructors in {class_name}:")
    print(top_constructors)

In [ ]:
# Heatmap of constructor performance by class
plt.figure(figsize=(15, 10))
# Select top constructors for better visualization
top_constructors = constructor_wins.head(15).index
constructor_class_subset = constructor_class.loc[:, constructor_class.loc[:, top_constructors].sum() > 0]
constructor_class_subset = constructor_class_subset[top_constructors]

sns.heatmap(constructor_class_subset, annot=True, fmt='d', cmap='YlOrRd', 
            cbar_kws={'label': 'Number of Wins'})
plt.title('Constructor Wins by Class (Top 15 Constructors)')
plt.xlabel('Constructor')
plt.ylabel('Class')
plt.tight_layout()
plt.show()

## 5. Rider Analysis

Let's analyze rider performance patterns.

In [ ]:
# Top riders by total wins
print("=== TOP RIDERS BY TOTAL WINS ===")
rider_wins = df['Rider'].value_counts()
print("Top 20 riders:")
print(rider_wins.head(20))

# Bar chart of top 20 riders
plt.figure(figsize=(14, 8))
top_20_riders = rider_wins.head(20)
plt.barh(range(len(top_20_riders)), top_20_riders.values, color='lightgreen')
plt.yticks(range(len(top_20_riders)), top_20_riders.index)
plt.xlabel('Number of Wins')
plt.title('Top 20 Riders by Total Wins (All Classes)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Riders by country
print("=== TOP COUNTRIES BY RACE WINS ===")
country_wins = df['Country'].value_counts()
print("Top 15 countries:")
print(country_wins.head(15))

# Pie chart for top countries
plt.figure(figsize=(12, 8))
top_10_countries = country_wins.head(10)
plt.pie(top_10_countries.values, labels=top_10_countries.index, autopct='%1.1f%%', 
        startangle=90)
plt.title('Top 10 Countries by Race Wins')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 6. Circuit Analysis

Let's analyze circuit patterns and characteristics.

In [ ]:
# Most popular circuits
print("=== MOST POPULAR CIRCUITS ===")
circuit_frequency = df['Circuit'].value_counts()
print("Top 15 circuits by number of races:")
print(circuit_frequency.head(15))

# Bar chart of top circuits
plt.figure(figsize=(14, 8))
top_15_circuits = circuit_frequency.head(15)
plt.bar(range(len(top_15_circuits)), top_15_circuits.values, color='salmon')
plt.xticks(range(len(top_15_circuits)), top_15_circuits.index, rotation=45, ha='right')
plt.xlabel('Circuit')
plt.ylabel('Number of Races')
plt.title('Top 15 Circuits by Race Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Circuit winner diversity analysis
print("=== CIRCUIT WINNER DIVERSITY ===")
circuit_diversity = {}
for circuit in df['Circuit'].unique():
    circuit_data = df[df['Circuit'] == circuit]
    unique_winners = circuit_data['Rider'].nunique()
    total_races = len(circuit_data)
    diversity_ratio = unique_winners / total_races if total_races > 0 else 0
    circuit_diversity[circuit] = {
        'total_races': total_races,
        'unique_winners': unique_winners,
        'diversity_ratio': diversity_ratio
    }

diversity_df = pd.DataFrame(circuit_diversity).T
diversity_df = diversity_df[diversity_df['total_races'] >= 5]  # Filter circuits with at least 5 races

print("\nCircuits with highest winner diversity (min 5 races):")
top_diversity = diversity_df.nlargest(10, 'diversity_ratio')
print(top_diversity[['total_races', 'unique_winners', 'diversity_ratio']])

## 7. Temporal Patterns

Let's analyze patterns over time.

In [ ]:
# Races per season over time
print("=== RACES PER SEASON EVOLUTION ===")
races_per_season = df['Season'].value_counts().sort_index()

plt.figure(figsize=(15, 8))
plt.plot(races_per_season.index, races_per_season.values, marker='o', linewidth=2)
plt.xlabel('Season')
plt.ylabel('Number of Races')
plt.title('Number of Races per Season Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAverage races per season: {races_per_season.mean():.1f}")
print(f"Minimum races in a season: {races_per_season.min()} (in {races_per_season.idxmin()})")
print(f"Maximum races in a season: {races_per_season.max()} (in {races_per_season.idxmax()})")

In [ ]:
# Dominant winners per season analysis
print("=== SEASON DOMINANCE ANALYSIS ===")
season_dominance = {}
for season in df['Season'].unique():
    season_data = df[df['Season'] == season]
    winner_counts = season_data['Rider'].value_counts()
    total_races = len(season_data)
    if len(winner_counts) > 0:
        most_wins = winner_counts.iloc[0]
        dominant_rider = winner_counts.index[0]
        dominance_percentage = (most_wins / total_races) * 100
        season_dominance[season] = {
            'dominant_rider': dominant_rider,
            'wins': most_wins,
            'total_races': total_races,
            'dominance_percentage': dominance_percentage
        }

dominance_df = pd.DataFrame(season_dominance).T
dominance_df = dominance_df.sort_values('dominance_percentage', ascending=False)

print("\nTop 10 most dominant seasons:")
print(dominance_df.head(10))

## 8. Regional Patterns Analysis

Let's analyze geographical patterns in the data.

In [ ]:
# Create continent mapping for analysis
continent_mapping = {
    'IT': 'Europe', 'ES': 'Europe', 'GB': 'Europe', 'FR': 'Europe', 'DE': 'Europe', 
    'NL': 'Europe', 'CH': 'Europe', 'AT': 'Europe', 'CZ': 'Europe', 'FI': 'Europe',
    'JP': 'Asia', 'TH': 'Asia', 'MY': 'Asia', 'CN': 'Asia', 'IN': 'Asia', 'ID': 'Asia',
    'AU': 'Oceania', 'NZ': 'Oceania',
    'US': 'North America', 'CA': 'North America',
    'BR': 'South America', 'AR': 'South America',
    'ZA': 'Africa', 'PT': 'Europe', 'SE': 'Europe', 'NO': 'Europe', 'DK': 'Europe',
    'BE': 'Europe', 'IE': 'Europe', 'LU': 'Europe', 'MC': 'Europe', 'SM': 'Europe',
    'VA': 'Europe', 'LI': 'Europe', 'AD': 'Europe', 'MT': 'Europe', 'CY': 'Europe'
}

# Apply continent mapping
df['Continent'] = df['Country'].map(continent_mapping)
df['Continent'] = df['Continent'].fillna('Other')

print("=== CONTINENTAL ANALYSIS ===")
continent_wins = df['Continent'].value_counts()
print("Wins by continent:")
print(continent_wins)

# Pie chart of continental distribution
plt.figure(figsize=(10, 8))
plt.pie(continent_wins.values, labels=continent_wins.index, autopct='%1.1f%%', 
        startangle=90)
plt.title('Distribution of Race Wins by Continent')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 9. Key Insights Summary

Based on the exploration of the race winners dataset:

### Dataset Characteristics
- Contains race winner data across multiple racing classes and seasons
- Comprehensive coverage of circuits, constructors, riders, and countries
- Rich temporal data allowing for trend analysis

### Racing Classes
- Multiple classes represented with varying frequency
- Evolution of class structure over time
- Different competitive dynamics per class

### Constructor Dominance
- Clear leaders in overall wins across all classes
- Class-specific constructor strengths
- Varying competitive landscapes by class

### Rider Performance
- Identification of most successful riders across all classes
- Strong geographical patterns in rider success
- Clear national dominance in certain periods

### Circuit Patterns
- Frequent vs. occasional circuits identified
- Winner diversity varies significantly by circuit
- Some circuits favor certain types of competitors

### Temporal Trends
- Season length has varied over time
- Periods of rider dominance clearly visible
- Continental representation has shifted over decades

This exploration provides the foundation for answering specific business questions about constructor dominance, temporal patterns, geographical analysis, and winner statistics.